# Baseline ChromaDB Retrieval Test

Verify that articles were ingested correctly and that semantic search returns relevant results.

In [7]:
import json
import chromadb
import requests

# Configuration
CHROMA_DIR = "../data/chroma_db"
COLLECTION_NAME = "tax_articles"
OLLAMA_URL = "http://localhost:11434/api/embed"
EMBED_MODEL = "qwen3-embedding:0.6b"
QUERY = "kriteria pengusaha kena pajak"
TOP_K = 3
CONTENT_PREVIEW_LEN = 200

## 1. Connect to ChromaDB and inspect the collection

In [8]:
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

total = collection.count()
print(f"Total articles in collection: {total}")

Total articles in collection: 34


## 2. Embed the query with Ollama

In [9]:
resp = requests.post(
    OLLAMA_URL,
    json={"model": EMBED_MODEL, "input": [QUERY]},
    timeout=30,
)
resp.raise_for_status()
query_vec = resp.json()["embeddings"][0]

print(f'Query : "{QUERY}"')
print(f"Embedding dim : {len(query_vec)}")

Query : "kriteria pengusaha kena pajak"
Embedding dim : 1024


## 3. Retrieve top-K results

In [10]:
results = collection.query(
    query_embeddings=[query_vec],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

docs  = results["documents"][0]
metas = results["metadatas"][0]
dists = results["distances"][0]

print(f"Top {TOP_K} results:")
print("=" * 60)
for rank, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
    refs = json.loads(meta.get("references", "[]"))
    print(f"[{rank}] Regulation : {meta['regulation_id']}")
    print(f"     Article    : Pasal {meta['article_number']}")
    print(f"     Distance   : {dist:.4f}  (cosine; lower = more similar)")
    print(f"     References : {refs if refs else '—'}")
    print(f"     Content    : {doc[:CONTENT_PREVIEW_LEN]!r}")
    print()

Top 3 results:
[1] Regulation : PMK 10 TAHUN 2026
     Article    : Pasal 34
     Distance   : 0.5372  (cosine; lower = more similar)
     References : —
     Content    : 'Peraturan Menteri ini mulai berlaku pada tanggal diundangkan. \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n\n- 32 - \n \n \nDitandatangani secara elektronik \nAgar \nsetiap \nor'

[2] Regulation : PMK 10 TAHUN 2026
     Article    : Pasal 13
     Distance   : 0.6165  (cosine; lower = more similar)
     References : ['12']
     Content    : '(1) \nAlokasi kinerja sebagaimana dimaksud dalam Pasal 12 \nayat (3) dihitung berdasarkan indikator: \na. \npenurunan tingkat kemiskinan; dan/atau \nb. \nketersediaan rencana aksi Daerah kelapa sawit \nberke'

[3] Regulation : PMK 10 TAHUN 2026
     Article    : Pasal 1
     Distance   : 0.6373  (cosine; lower = more similar)
     References : —
     Content    : 'Dalam Peraturan Menteri ini yang dimaksud dengan: